In [ ]:
from pathlib import Path

import pandas as pd

# ======================== User configuration ========================
PANEL = Path("../outputs/final_panel/panel_final_main_local_p90_with_p95_2016_2023.csv")
OUTDIR = Path("../outputs/figures/fig1")
YEARS = range(2016, 2024)
MIN_CELLS_WITHIN_CITY = 3
EXAMPLE_CITIES = ["Lima_Peru", "Dhaka_Bangladesh"]
EXAMPLE_YEAR = 2023
# ==========================================================

OUTDIR.mkdir(parents=True, exist_ok=True)
TYPES = ["day", "night", "compound"]
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)


def spearman(a, b):
    a, b = pd.Series(a).reset_index(drop=True), pd.Series(b).reset_index(drop=True)
    return a.rank().corr(b.rank())


stats = []


def report(paragraph, name, value, note=""):
    stats.append(dict(paragraph=paragraph, statistic=name, value=value, note=note))


# ---------------- Data ----------------
raw = pd.read_csv(PANEL, low_memory=False)
raw = raw[raw.year.isin(YEARS)]
complete = raw.heatwave_annual_complete.astype(str).str.lower().eq("true")
d = raw[complete].copy()

agg = {"pop": ("pop_total", "sum")}
for t in TYPES:
    agg[f"exp_{t}"] = (f"exposure_people_days_{t}", "sum")
    agg[f"hwd_{t}"] = (f"HW_days_{t}", "mean")
cy = d.groupby(["city", "region", "year"], as_index=False).agg(**agg)
cy["exp_total"] = cy[[f"exp_{t}" for t in TYPES]].sum(axis=1)
ca = cy.groupby(["city", "region"], as_index=False)[[c for c in cy.columns if c.startswith(("exp_", "hwd_"))]].mean()
for c in [c for c in ca.columns if c.startswith("exp_")]:
    ca[c] = ca[c] / 1e6

# ---------------- Section 1: interannual variation ----------------
for t in TYPES:
    n_years = cy.assign(pos=cy[f"exp_{t}"] > 0).groupby("city").pos.sum()
    report(1, f"{t}: years with exposure (city median)", float(n_years.median()), f"/ {len(YEARS)} years")
    all_years = sorted(n_years[n_years == len(YEARS)].index)
    report(1, f"{t}: number of cities with exposure in every year", len(all_years), ", ".join(all_years) if len(all_years) <= 3 else "")
for t in TYPES:
    g = cy.groupby("city")[f"exp_{t}"]
    share = (g.max() / g.sum())[g.sum() > 0]
    report(1, f"{t}: share of 8-year total exposure in the peak year (city median)", float(share.median()),
           f"IQR {share.quantile(.25):.2f}–{share.quantile(.75):.2f}")

annual = cy.groupby("year")[[f"exp_{t}" for t in TYPES] + ["exp_total"]].sum() / 1e9
other = annual.drop(index=EXAMPLE_YEAR)
report(1, f"{EXAMPLE_YEAR} total exposure, three types combined (billion person-days)", float(annual.loc[EXAMPLE_YEAR, "exp_total"]))
report(1, "Total exposure in the highest earlier year (billion person-days)", float(other.exp_total.max()), f"year {int(other.exp_total.idxmax())}")
report(1, f"{EXAMPLE_YEAR} / highest earlier year", float(annual.loc[EXAMPLE_YEAR, "exp_total"] / other.exp_total.max()))

for c in EXAMPLE_CITIES:
    x = cy[cy.city.eq(c)].set_index("year")
    report(1, f"{c} {EXAMPLE_YEAR} mean grid-cell compound HWD (days)", float(x.loc[EXAMPLE_YEAR, "hwd_compound"]))
    report(1, f"{c} {EXAMPLE_YEAR} compound exposure (million person-days)", float(x.loc[EXAMPLE_YEAR, "exp_compound"] / 1e6))
    report(1, f"{c} {EXAMPLE_YEAR} share of 8-year compound exposure", float(x.loc[EXAMPLE_YEAR, "exp_compound"] / x.exp_compound.sum()))
    zero = [int(y) for y in x.index if y < EXAMPLE_YEAR and x.loc[y, "exp_compound"] == 0]
    report(1, f"{c}: years before {EXAMPLE_YEAR} with zero compound exposure", len(zero), str(zero))

# ---------------- Section 2: differences among cities and regions ----------------
top = ca.sort_values("exp_total", ascending=False).reset_index(drop=True)
lead, tier = top.iloc[:2], top.iloc[2:5]
for _, r in lead.iterrows():
    report(2, f"{r.city} mean annual exposure, three types combined (million person-days/year)", float(r.exp_total))
report(2, "Second tier (ranks 3–5) range (million person-days/year)", f"{tier.exp_total.min():.1f}–{tier.exp_total.max():.1f}", ", ".join(tier.city))
report(2, "Ratio range, top two / second tier", f"{lead.exp_total.min() / tier.exp_total.max():.2f}–{lead.exp_total.max() / tier.exp_total.min():.2f}")

reg = ca.groupby("region")[[f"exp_{t}" for t in TYPES] + [f"hwd_{t}" for t in TYPES]].median()
reg.insert(0, "n_cities", ca.groupby("region").size())
for t in TYPES:
    report(2, f"{t}: regional median exposure, lowest–highest", f"{reg[f'exp_{t}'].min():.2f} ({reg[f'exp_{t}'].idxmin()}) – "
                                               f"{reg[f'exp_{t}'].max():.2f} ({reg[f'exp_{t}'].idxmax()})")
la, sa = reg.loc["Latin America"], reg.loc["South Asia"]
report(2, "Latin America compound HWD median (days/year)", float(la.hwd_compound))
report(2, "South Asia compound HWD median (days/year)", float(sa.hwd_compound))
report(2, "Latin America compound HWD exceeds South Asia by", float(la.hwd_compound / sa.hwd_compound - 1))
report(2, "Median compound exposure, Latin America / South Asia", float(la.exp_compound / sa.exp_compound))
report(2, "Region with the highest median compound HWD", reg.hwd_compound.idxmax())

# ---------------- Section 3: scale dependence ----------------
report(3, "Number of city-years", len(cy))
for t in TYPES:
    report(3, f"{t}: city-year ρ(exposure, HWD)", spearman(cy[f"hwd_{t}"], cy[f"exp_{t}"]))
    report(3, f"{t}: city-year ρ(exposure, population)", spearman(cy["pop"], cy[f"exp_{t}"]))

rows = []
for t in TYPES:
    for city, g in d.groupby("city"):
        cell = g.groupby("cell_id").agg(exp=(f"exposure_people_days_{t}", "mean"), pop=("pop_total", "mean"),
                                        hwd=(f"HW_days_{t}", "mean")).dropna()
        if len(cell) < MIN_CELLS_WITHIN_CITY or cell.exp.sum() <= 0:
            continue
        rows.append(dict(heatwave_type=t, city=city, n_cells=len(cell),
                         rho_population=spearman(cell.exp, cell["pop"]), rho_hwd=spearman(cell.exp, cell.hwd)))
within = pd.DataFrame(rows)
for t in TYPES:
    s = within[within.heatwave_type.eq(t)]
    q = lambda v: f"{v.median():.2f} [{v.quantile(.25):.2f}–{v.quantile(.75):.2f}]"
    report(3, f"{t}: within-city ρ(exposure, population), median [IQR]", q(s.rho_population), f"n = {len(s)} cities")
    report(3, f"{t}: within-city ρ(exposure, HWD), median [IQR]", q(s.rho_hwd))
    report(3, f"{t}: number of cities where the population correlation > the HWD correlation", f"{int((s.rho_population > s.rho_hwd).sum())} / {len(s)}")

# ---------------- Save ----------------
pd.DataFrame(stats).to_csv(OUTDIR / "results_part1_statistics.csv", index=False, encoding="utf-8-sig")
cy.to_csv(OUTDIR / "city_year_exposure.csv", index=False)
ca.to_csv(OUTDIR / "city_mean_annual_exposure.csv", index=False)
reg.to_csv(OUTDIR / "region_medians.csv")
within.to_csv(OUTDIR / "within_city_correlations.csv", index=False)
annual.to_csv(OUTDIR / "annual_total_exposure_billion.csv")
print("Output:", OUTDIR)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


# ===================== paths =====================
PANEL = Path("../outputs/final_panel/panel_final_main_local_p90_with_p95_2016_2023.csv")

OUT_DIR = Path("../outputs/figures/fig1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2016, 2024))


# ===================== config =====================
EXPOSURE_COLS = {
    "Daytime": "exposure_people_days_day",
    "Nighttime": "exposure_people_days_night",
    "Compound": "exposure_people_days_compound",
}

TYPE_COLORS = {
    "Daytime": "#E8A33D",
    "Nighttime": "#4E8AC2",
    "Compound": "#A43A33",
}


# ===================== data =====================
df = pd.read_csv(PANEL, low_memory=False)
df = df[(df["pct"] == "p90") & (df["year"].isin(YEARS))].copy()
df = df[df["heatwave_annual_complete"].astype(str).str.lower().eq("true")].copy()

annual = (
    df.groupby("year", as_index=True)[list(EXPOSURE_COLS.values())]
    .sum()
    .rename(columns={v: k for k, v in EXPOSURE_COLS.items()})
    / 1_000_000_000
)

annual = annual.reindex(YEARS)

annual.to_csv(
    OUT_DIR / "fig1_top_annual_total_exposure_2016_2023.csv"
)


# ===================== plot =====================
plt.rcParams.update(
    {
        "font.family": "Arial",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.9,
    }
)

fig, ax = plt.subplots(figsize=(9.8, 1.85), dpi=300)

for label, color in TYPE_COLORS.items():
    ax.plot(
        annual.index,
        annual[label],
        color=color,
        marker="o",
        markersize=4.8,
        linewidth=2.15,
        label=label,
        zorder=3,
    )

ax.set_xlim(min(YEARS) - 0.18, max(YEARS) + 0.18)
ax.set_ylim(0, max(annual.max()) * 1.12)

ax.set_xticks(YEARS)
ax.set_xlabel("")

ax.set_ylabel(
    "Heatwave population exposure\n(billion person-days)",
    fontsize=9.8,
)

ax.grid(axis="y", color="#D9D9D9", linewidth=0.7)
ax.grid(axis="x", visible=False)

ax.tick_params(axis="both", labelsize=9.2, length=3.2, color="#333333")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#333333")
ax.spines["bottom"].set_color("#333333")
ax.spines["left"].set_linewidth(0.9)
ax.spines["bottom"].set_linewidth(0.9)

ax.legend(
    frameon=False,
    ncol=3,
    loc="upper left",
    bbox_to_anchor=(0.075, 1.18),
    fontsize=9.6,
    handlelength=2.3,
    columnspacing=1.65,
    borderaxespad=0,
)

fig.tight_layout(pad=0.35)


# ===================== save =====================
png = OUT_DIR / "fig1_top_annual_total_exposure_compact_2016_2023.png"
pdf = OUT_DIR / "fig1_top_annual_total_exposure_compact_2016_2023.pdf"

fig.savefig(png, dpi=300, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")

plt.show()

print("Output:", OUT_DIR)

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import BoundaryNorm, LinearSegmentedColormap


# ===================== paths =====================
PANEL = Path("../outputs/final_panel/panel_final_main_local_p90_with_p95_2016_2023.csv")

LOCAL_PHYSICAL = Path(
    "../data/natural_earth"
)
LOCAL_LAND = LOCAL_PHYSICAL / "ne_110m_land.shp"
LOCAL_COAST = LOCAL_PHYSICAL / "ne_110m_coastline.shp"

OUT_DIR = Path("../outputs/figures/fig1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2016, 2024))


# ===================== columns =====================
EXPOSURE_COLS = {
    "Daytime": "exposure_people_days_day",
    "Nighttime": "exposure_people_days_night",
    "Compound": "exposure_people_days_compound",
}

HWDAY_COLS = {
    "Daytime": "HW_days_day",
    "Nighttime": "HW_days_night",
    "Compound": "HW_days_compound",
}


# ===================== palette =====================
EXPOSURE_PALETTE = [
    "#FFF8B8",
    "#FF9468",
    "#F89040",
    "#E04868",
    "#902880",
    "#704890",
    "#401070",
]

EXPOSURE_CMAP = LinearSegmentedColormap.from_list(
    "custom_exposure",
    EXPOSURE_PALETTE,
    N=256,
)


# ===================== helpers =====================
def fmt_lon(x):
    if x == 0:
        return "0"
    return f"{abs(int(x))}W" if x < 0 else f"{abs(int(x))}E"


def fmt_lat(y):
    if y == 0:
        return "0"
    return f"{abs(int(y))}S" if y < 0 else f"{abs(int(y))}N"


def nice_bounds(vmax):
    if vmax > 250:
        step = 50
    elif vmax > 100:
        step = 20
    elif vmax > 50:
        step = 10
    else:
        step = 5

    top = ((int(vmax) // step) + 1) * step
    return np.arange(0, top + step, step)


def read_basemap():
    if not LOCAL_LAND.exists():
        raise FileNotFoundError(f"Land shapefile not found: {LOCAL_LAND}")

    if not LOCAL_COAST.exists():
        raise FileNotFoundError(f"Coastline shapefile not found: {LOCAL_COAST}")

    land = gpd.read_file(LOCAL_LAND)
    coast = gpd.read_file(LOCAL_COAST)

    return land, coast


def build_city_means():
    panel = pd.read_csv(PANEL, low_memory=False)
    panel = panel[(panel["pct"] == "p90") & (panel["year"].isin(YEARS))].copy()
    panel = panel[panel["heatwave_annual_complete"].astype(str).str.lower().eq("true")].copy()

    annual = (
        panel.groupby(["city", "year"], as_index=False)
        .agg(
            lat=("lat", "mean"),
            lon=("lon", "mean"),
            **{col: (col, "sum") for col in EXPOSURE_COLS.values()},
            **{col: (col, "mean") for col in HWDAY_COLS.values()},
        )
    )

    rows = []

    for label, exp_col in EXPOSURE_COLS.items():
        hw_col = HWDAY_COLS[label]

        out = (
            annual.groupby("city", as_index=False)
            .agg(
                lat=("lat", "mean"),
                lon=("lon", "mean"),
                mean_exposure_people_days=(exp_col, "mean"),
                mean_hw_days=(hw_col, "mean"),
            )
        )

        out["heatwave_type"] = label
        out["mean_exposure_million_people_days"] = (
            out["mean_exposure_people_days"] / 1_000_000
        )

        rows.append(out)

    return pd.concat(rows, ignore_index=True)


# ===================== build data =====================
land, coast = read_basemap()
city_means = build_city_means()

city_means.to_csv(
    OUT_DIR / "fig1_mean_maps_city_means_2016_2023.csv",
    index=False,
)


# ===================== plot =====================
sns.set_theme(style="white", context="paper", font="Arial")

hwdays_max = float(np.nanpercentile(city_means["mean_hw_days"], 95))
hwdays_max = max(hwdays_max, 1.0)

fig = plt.figure(figsize=(9.8, 9.2), constrained_layout=False)

gs = fig.add_gridspec(
    3,
    2,
    width_ratios=[38, 1.15],
    hspace=0.34,
    wspace=0.035,
)

axes = [fig.add_subplot(gs[i, 0]) for i in range(3)]
caxes = [fig.add_subplot(gs[i, 1]) for i in range(3)]

fig.subplots_adjust(
    left=0.08,
    right=0.955,
    top=0.94,
    bottom=0.075,
)

for i, (ax, hw_type) in enumerate(zip(axes, EXPOSURE_COLS.keys())):
    row = city_means[city_means["heatwave_type"] == hw_type].copy()

    values = row["mean_exposure_million_people_days"]
    bounds = nice_bounds(float(values.max()))
    norm = BoundaryNorm(bounds, EXPOSURE_CMAP.N, clip=True)

    land.plot(
        ax=ax,
        color="#D4D4D4",
        edgecolor="none",
        linewidth=0,
        zorder=0,
    )

    coast.plot(
        ax=ax,
        color="white",
        linewidth=0.55,
        zorder=1,
    )

    size_scale = np.sqrt(row["mean_hw_days"].clip(upper=hwdays_max) / hwdays_max)
    sizes = 28 + 64 * size_scale

    order = np.argsort(-sizes.to_numpy())

    scatter = ax.scatter(
        row["lon"].to_numpy()[order],
        row["lat"].to_numpy()[order],
        c=values.to_numpy()[order],
        cmap=EXPOSURE_CMAP,
        norm=norm,
        s=sizes.to_numpy()[order],
        marker="o",
        edgecolor="#6F1231",
        linewidth=1.15,
        alpha=0.88,
        zorder=5,
    )

    ax.set_xlim(-125, 150)
    ax.set_ylim(-45, 45)

    ax.set_xticks([-100, -50, 0, 50, 100, 150])
    ax.set_yticks([-45, -30, 0, 30, 45])

    ax.set_xticklabels([fmt_lon(x) for x in ax.get_xticks()], fontsize=10)
    ax.set_yticklabels([fmt_lat(y) for y in ax.get_yticks()], fontsize=10)

    ax.set_xlabel("Longitude", fontsize=11)
    ax.set_ylabel("Latitude", fontsize=11)

    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.1)

    cbar = fig.colorbar(
        scatter,
        cax=caxes[i],
        boundaries=bounds,
        ticks=bounds,
        extend="max",
    )

    cbar.ax.set_title("M people-days", fontsize=8, pad=5)
    cbar.ax.tick_params(labelsize=8, length=3)


# ===================== save =====================
png = OUT_DIR / "fig1_mean_maps_2016_2023.png"
pdf = OUT_DIR / "fig1_mean_maps_2016_2023.pdf"

fig.savefig(png, dpi=300, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")

plt.show()

print("Output:", OUT_DIR)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ===================== paths =====================
PANEL = Path("../outputs/final_panel/panel_final_main_local_p90_with_p95_2016_2023.csv")

OUT_DIR = Path("../outputs/figures/fig1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2016, 2024))


# ===================== config =====================
TYPE_CONFIG = {
    "day": {
        "label": "Daytime",
        "exp_col": "exposure_people_days_day",
        "xmax": 350,
        "xticks": [0, 100, 200, 300],
    },
    "night": {
        "label": "Nighttime",
        "exp_col": "exposure_people_days_night",
        "xmax": 140,
        "xticks": [0, 50, 100],
    },
    "compound": {
        "label": "Compound",
        "exp_col": "exposure_people_days_compound",
        "xmax": 120,
        "xticks": [0, 40, 80, 120],
    },
}

REGION_ORDER = ["Africa", "South Asia", "Southeast Asia", "Latin America"]

REGION_COLORS = {
    "Africa": "#D9822B",
    "South Asia": "#D84A61",
    "Southeast Asia": "#4D8FCB",
    "Latin America": "#4C9A5C",
}


# ===================== data =====================
df = pd.read_csv(PANEL, low_memory=False)
df = df[(df["pct"] == "p90") & (df["year"].isin(YEARS))].copy()
df = df[df["heatwave_annual_complete"].astype(str).str.lower().eq("true")].copy()

city_year = (
    df.groupby(["city", "year", "region"], as_index=False)
    .agg(
        **{
            cfg["exp_col"]: (cfg["exp_col"], "sum")
            for cfg in TYPE_CONFIG.values()
        }
    )
)

city_mean = (
    city_year.groupby(["city", "region"], as_index=False)
    .agg(
        **{
            cfg["exp_col"]: (cfg["exp_col"], "mean")
            for cfg in TYPE_CONFIG.values()
        }
    )
)

for cfg in TYPE_CONFIG.values():
    city_mean[cfg["exp_col"] + "_million"] = city_mean[cfg["exp_col"]] / 1_000_000

city_mean.to_csv(
    OUT_DIR / "fig1_region_boxplot_city_means_2016_2023.csv",
    index=False,
)


# ===================== helper =====================
def draw_region_boxplot_clean(ax, data, type_key):
    cfg = TYPE_CONFIG[type_key]
    value_col = cfg["exp_col"] + "_million"

    positions = np.arange(len(REGION_ORDER), 0, -1)

    grouped = [
        data.loc[data["region"] == region, value_col].dropna().to_numpy()
        for region in REGION_ORDER
    ]

    bp = ax.boxplot(
        grouped,
        vert=False,
        positions=positions,
        widths=0.56,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "#222222", "linewidth": 1.05},
        whiskerprops={"color": "#555555", "linewidth": 0.72},
        capprops={"color": "#555555", "linewidth": 0.72},
    )

    for box, region in zip(bp["boxes"], REGION_ORDER):
        box.set_facecolor(REGION_COLORS[region])
        box.set_edgecolor(REGION_COLORS[region])
        box.set_alpha(0.34)
        box.set_linewidth(0.95)

    rng = np.random.default_rng(2026)

    for pos, region, values in zip(positions, REGION_ORDER, grouped):
        jitter = rng.normal(0, 0.043, len(values))

        ax.scatter(
            values,
            pos + jitter,
            s=10,
            color=REGION_COLORS[region],
            alpha=0.78,
            linewidth=0,
            zorder=3,
        )

    ax.set_xlim(0, cfg["xmax"])
    ax.set_xticks(cfg["xticks"])
    ax.set_ylim(0.35, len(REGION_ORDER) + 0.65)
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.grid(axis="x", color="#E7E7E7", linewidth=0.5)
    ax.tick_params(axis="x", labelsize=7.7, length=2.8, color="#333333", pad=2)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#333333")
        spine.set_linewidth(0.78)


# ===================== plot =====================
plt.rcParams.update(
    {
        "font.family": "Arial",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.78,
    }
)

fig, axes = plt.subplots(
    3,
    1,
    figsize=(2.35, 8.95),
    dpi=300,
    constrained_layout=False,
)

fig.subplots_adjust(
    left=0.12,
    right=0.97,
    top=0.985,
    bottom=0.055,
    hspace=0.50,
)

for ax, type_key in zip(axes, TYPE_CONFIG):
    draw_region_boxplot_clean(ax, city_mean, type_key)


# ===================== save =====================
png = OUT_DIR / "fig1_region_boxplots_2016_2023.png"
pdf = OUT_DIR / "fig1_region_boxplots_2016_2023.pdf"

fig.savefig(png, dpi=300, bbox_inches="tight")
fig.savefig(pdf, bbox_inches="tight")

plt.show()

print("Output:", OUT_DIR)

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

ROOT = Path("..")
PANEL = Path("../outputs/final_panel/panel_final_main_local_p90_with_p95_2016_2023.csv")
OUT = ROOT / "outputs" / "figures" / "fig1"
OUT.mkdir(parents=True, exist_ok=True)

YEARS = (2016, 2023)
MIN_CELLS = 3

TYPES = {
    "day": dict(label="Daytime",
                exposure="exposure_people_days_day",
                hwd="HW_days_day",
                color="#E4A93C"),
    "night": dict(label="Nighttime",
                  exposure="exposure_people_days_night",
                  hwd="HW_days_night",
                  color="#3D6E9E"),
    "compound": dict(label="Compound",
                     exposure="exposure_people_days_compound",
                     hwd="HW_days_compound",
                     color="#8C2E3B"),
}

INK, SUBTLE, GRID = "#222222", "#666666", "#E8E8E8"
BLUE_TINT, WARM_TINT = "#3D6E9E", "#8C6A2E"

mpl.rcParams.update(mpl.rcParamsDefault)
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.7,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
    "xtick.major.size": 2.6,
    "ytick.major.size": 2.6,
    "legend.frameon": False,
})


def spearman(x, y):
    x, y = pd.Series(np.asarray(x, float)), pd.Series(np.asarray(y, float))
    ok = x.notna() & y.notna()
    if ok.sum() < 3:
        return np.nan
    return x[ok].rank().corr(y[ok].rank())


df = pd.read_csv(PANEL, low_memory=False)
df = df[df["pct"].eq("p90") & df["year"].between(*YEARS)].copy()
df = df[df["heatwave_annual_complete"].astype(str).str.lower().eq("true")].copy()

agg = {"population": ("pop_total", "sum")}
for k, c in TYPES.items():
    agg[f"exposure_{k}"] = (c["exposure"], "sum")
    agg[f"hwd_{k}"] = (c["hwd"], "mean")

city_year = df.groupby(["city", "year"], as_index=False).agg(**agg)

rows = []
for k, c in TYPES.items():
    for city, g in df.groupby("city"):
        cm = (g.groupby("cell_id", as_index=False)
              .agg(exposure=(c["exposure"], "mean"),
                   population=("pop_total", "mean"),
                   hwd=(c["hwd"], "mean"))
              .dropna())
        if len(cm) < MIN_CELLS or cm["exposure"].nunique() < 2:
            continue
        rows.append(dict(city=city, heatwave_type=k, n_cells=len(cm),
                         rho_population=spearman(cm.exposure, cm.population),
                         rho_hwd=spearman(cm.exposure, cm.hwd)))
within = pd.DataFrame(rows)

fig = plt.figure(figsize=(7.2, 3.5), dpi=300)
gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.05],
                      left=0.085, right=0.985, bottom=0.165, top=0.855,
                      wspace=0.34)
ax_e = fig.add_subplot(gs[0, 0])
ax_f = fig.add_subplot(gs[0, 1])

fig.legend(
    handles=[Line2D([0], [0], marker="o", linestyle="none", markersize=5,
                    markerfacecolor=c["color"], markeredgecolor="none",
                    label=c["label"]) for c in TYPES.values()],
    loc="upper center", bbox_to_anchor=(0.5, 1.005), ncol=3,
    fontsize=8, handletextpad=0.35, columnspacing=1.6,
)

for ax, letter in ((ax_e, "e"), (ax_f, "f")):
    ax.text(-0.155, 1.06, letter, transform=ax.transAxes,
            fontsize=10, fontweight="bold", va="bottom", ha="left")
    ax.grid(color=GRID, linewidth=0.55, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

stats = []
for k, c in TYPES.items():
    x = city_year[f"hwd_{k}"].to_numpy(float)
    y = city_year[f"exposure_{k}"].to_numpy(float) / 1e6
    p = city_year["population"].to_numpy(float)
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(p)
    ax_e.scatter(x[ok], y[ok], s=13, alpha=0.42, color=c["color"],
                 edgecolors="none", rasterized=True, zorder=3)
    stats.append((spearman(x[ok], y[ok]), spearman(p[ok], y[ok])))

ax_e.set_xscale("symlog", linthresh=1, linscale=0.45)
ax_e.set_yscale("symlog", linthresh=1, linscale=0.45)
ax_e.set_xlim(-0.12, 90)
ax_e.set_ylim(-0.12, 1600)
ax_e.set_xticks([0, 1, 3, 10, 30])
ax_e.set_yticks([0, 1, 10, 100, 1000])
fmt = FuncFormatter(lambda v, _: f"{v:g}")
ax_e.xaxis.set_major_formatter(fmt)
ax_e.yaxis.set_major_formatter(fmt)

ax_e.set_xlabel("Mean grid-cell heatwave days", labelpad=3)
ax_e.set_ylabel("City-year exposure\n(million person-days)", labelpad=3)
ax_e.set_title("City-year scale", loc="left", fontsize=8.5,
               color=SUBTLE, pad=5)

ax_e.text(0.035, 0.965, "ρ with exposure", transform=ax_e.transAxes,
          fontsize=7, color=SUBTLE, va="top", ha="left")
ax_e.text(0.035, 0.895, "heat   pop.", transform=ax_e.transAxes,
          fontsize=6.5, color=SUBTLE, va="top", ha="left")
for i, (rh, rp) in enumerate(stats):
    ax_e.text(0.035, 0.835 - i * 0.075, f"{rh:.2f}   {rp:.2f}",
              transform=ax_e.transAxes, fontsize=7,
              color=TYPES[list(TYPES)[i]]["color"], va="top", ha="left")

LO, HI = -0.4, 1.05
ax_f.fill_between([LO, HI], [LO, HI], HI, color=BLUE_TINT,
                  alpha=0.045, lw=0, zorder=1)
ax_f.fill_between([LO, HI], LO, [LO, HI], color=WARM_TINT,
                  alpha=0.045, lw=0, zorder=1)
ax_f.plot([LO, HI], [LO, HI], ls=(0, (4, 3)), color="#999999",
          lw=0.8, zorder=2)

for k, c in TYPES.items():
    s = within[within.heatwave_type.eq(k)]
    ax_f.scatter(s.rho_hwd, s.rho_population, s=24, alpha=0.62,
                 color=c["color"], edgecolors="white", linewidths=0.4,
                 rasterized=True, zorder=3)

ax_f.set_xlim(LO, HI)
ax_f.set_ylim(LO, HI)
ax_f.set_aspect("equal", adjustable="box")
ax_f.set_xticks([-0.25, 0, 0.25, 0.5, 0.75, 1.0])
ax_f.set_yticks([-0.25, 0, 0.25, 0.5, 0.75, 1.0])
ax_f.set_xlabel("ρ (exposure, heatwave days)", labelpad=3)
ax_f.set_ylabel("ρ (exposure, population)", labelpad=3)
ax_f.set_title("Within-city scale", loc="left", fontsize=8.5,
               color=SUBTLE, pad=5)

fig.savefig(OUT / "fig1_panels_e_f.png", dpi=600, bbox_inches="tight")
fig.savefig(OUT / "fig1_panels_e_f.pdf", bbox_inches="tight")
plt.show()